# Moyenne, broadcasting et masque : préparer les valeurs sans perdre les cibles

Centrer les mêmes colonnes, puis sélectionner des extraits avec leurs annotations.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#selection-diffusion-reduction).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Nous savons donner des lots au modèle. Avant de changer ses couches, il faut aussi contrôler la préparation des mesures. Certains entraînements utilisent des écarts à une référence : essayons de soustraire la moyenne de chaque colonne, sans modifier X.

Nous gardons les trois extraits et leurs colonnes RMS puis facteur de crête. Les valeurs sont choisies. Centrer peut aider une optimisation, mais ce n’est pas une obligation de PyTorch ; la synthèse du cours gardera X brut pour retrouver la première expérience.

Une valeur manquante NaN contamine une moyenne. Le programme vérifie X avant mean : isfinite teste chaque case, all demande que toutes soient finies, puis item extrait le booléen Python. Un False invite à retrouver la mesure invalide, plutôt qu’à modifier le modèle.

dim=0 réunit les trois extraits de chaque colonne : (0,10+0,20+0,15)/3=0,15 ; (1,5+4,5+2,5)/3≈2,83. keepdim=True garde l’axe réduit avec une taille 1 : référence (1,2). dim=1 ferait au contraire une moyenne dans chaque ligne.

La colonne RMS soustrait toujours 0,15 ; l’autre soustrait environ 2,83. Les formes s’alignent depuis la droite ; un axe absent est traité comme une taille 1. Ainsi, 2 correspond à 2, puis la taille 1 se diffuse sur trois lignes. Un vecteur (3,) n’indiquerait pas « une valeur par ligne » : sa taille rencontrerait d’abord les deux colonnes et serait incompatible.

Supposons ensuite que notre expérience ne garde que les facteurs de crête strictement supérieurs à 2. Le filtre doit être décidé sur les mesures brutes : après centrage, le seuil n’aurait plus le même sens.

| Extrait | Crête brute | Ligne centrée, arrondie | Cible |
| --- | --- | --- | --- |
| B | 4,5 | [0,05 ; 1,67] | 1 |
| C | 2,5 | [0 ; −0,33] | 0 |

masque = [False, True, True] ; cibles_retenues = [1,0].

X[:,1] lit la seconde colonne ; sa comparaison produit une décision par extrait. Réutiliser le même masque sur les entrées et les cibles préserve les couples. Filtrer seulement les entrées donnerait deux lignes face à trois annotations.

Dans un vrai protocole, la référence est calculée sur l’entraînement, puis réutilisée sur les exemples réservés. La recalculer séparément changerait la transformation ; la calculer sur toutes les données ferait entrer la réserve dans la préparation.

## Démonstration 1 · Centrer puis sélectionner les mêmes lignes

Retrouver les objets et exécuter les fragments du chapitre dans leur contexte complet.

### Préparer les bibliothèques

Les valeurs et classes sont celles du fil conducteur. Le diagnostic interrompt la préparation si une case est NaN ou infinie.

In [3]:
import torch


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [4]:
X = torch.tensor([[.1, 1.5], [.2, 4.5], [.15, 2.5]])
cibles = torch.tensor([0, 1, 0], dtype=torch.long)
valeurs_finies = torch.isfinite(X)
controle_fini = valeurs_finies.all().item()


In [5]:
observer('X', X)
observer('cibles', cibles)
observer('valeurs_finies', valeurs_finies)
observer('controle_fini', controle_fini)

X forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[0.10000000149011612, 1.5], [0.20000000298023224, 4.5], [0.15000000596046448, 2.5]]
cibles forme (3,) type torch.int64 requires_grad False
  valeurs : [0, 1, 0]
valeurs_finies forme (3, 2) type torch.bool requires_grad False
  valeurs : [[True, True], [True, True], [True, True]]
controle_fini = True


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [6]:
if not controle_fini:
    raise ValueError("Les mesures doivent être finies avant de calculer leur moyenne.")


### Calculer l’état suivant

mean réduit les lignes ; la taille 1 conservée permet de soustraire les deux moyennes à chaque extrait.

In [7]:
reference = X.mean(dim=0, keepdim=True)
centres = X - reference


In [8]:
observer('reference', reference)
observer('centres', centres)

reference forme (1, 2) type torch.float32 requires_grad False
  valeurs : [[0.15000000596046448, 2.8333332538604736]]
centres forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[-0.05000000447034836, -1.3333332538604736], [0.04999999701976776, 1.6666667461395264], [0.0, -0.33333325386047363]]


### Calculer l’état suivant

La comparaison porte sur le facteur de crête avant centrage.

In [9]:
cretes = X[:, 1]
masque = cretes > 2.


In [10]:
observer('cretes', cretes)
observer('masque', masque)

cretes forme (3,) type torch.float32 requires_grad False
  valeurs : [1.5, 4.5, 2.5]
masque forme (3,) type torch.bool requires_grad False
  valeurs : [False, True, True]


### Calculer l’état suivant

Les mêmes positions sélectionnent les entrées transformées et leurs cibles.

In [11]:
retenus = centres[masque]
cibles_retenues = cibles[masque]


In [12]:
observer('retenus', retenus)
observer('cibles_retenues', cibles_retenues)

retenus forme (2, 2) type torch.float32 requires_grad False
  valeurs : [[0.04999999701976776, 1.6666667461395264], [0.0, -0.33333325386047363]]
cibles_retenues forme (2,) type torch.int64 requires_grad False
  valeurs : [1, 0]


### Lire les sorties

Les impressions lisent les résultats sans modifier X.

In [13]:
print("reference :", reference, "forme", tuple(reference.shape))
print("centres :", centres)
print("masque :", masque.tolist())
print("lignes retenues :", retenus)


reference : tensor([[0.1500, 2.8333]]) forme (1, 2)
centres : tensor([[-0.0500, -1.3333],
        [ 0.0500,  1.6667],
        [ 0.0000, -0.3333]])
masque : [False, True, True]
lignes retenues : tensor([[ 0.0500,  1.6667],
        [ 0.0000, -0.3333]])


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [14]:
print("cibles retenues :", cibles_retenues.tolist())
print("X reste brut :", X)


cibles retenues : [1, 0]
X reste brut : tensor([[0.1000, 1.5000],
        [0.2000, 4.5000],
        [0.1500, 2.5000]])


In [15]:
assert bool(tuple(reference.shape) == (1,2) and torch.allclose(reference, torch.tensor([[.15,17/6]]), atol=1e-6)), 'selection-diffusion-reduction/main: tuple(reference.shape) == (1,2) and torch.allclose(reference, torch.tensor([[.15,17/6]]), atol=1e-6)'
assert bool(torch.allclose(centres, torch.tensor([[-.05,-4/3],[.05,5/3],[0.,-1/3]]), atol=1e-6)), 'selection-diffusion-reduction/main: torch.allclose(centres, torch.tensor([[-.05,-4/3],[.05,5/3],[0.,-1/3]]), atol=1e-6)'
assert bool(masque.tolist() == [False,True,True] and cibles_retenues.tolist() == [1,0]), 'selection-diffusion-reduction/main: masque.tolist() == [False,True,True] and cibles_retenues.tolist() == [1,0]'
assert bool(tuple(retenus.shape) == (2,2) and torch.equal(X, torch.tensor([[.1,1.5],[.2,4.5],[.15,2.5]]))), 'selection-diffusion-reduction/main: tuple(retenus.shape) == (2,2) and torch.equal(X, torch.tensor([[.1,1.5],[.2,4.5],[.15,2.5]]))'
assert bool(controle_fini and not torch.isfinite(torch.tensor([.1,float("nan")])).all().item()), 'selection-diffusion-reduction/main: controle_fini and not torch.isfinite(torch.tensor([.1,float("nan")])).all().item()'
print("Contrôles numériques de la démonstration réussis.")

Contrôles numériques de la démonstration réussis.


## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Centrez chaque descripteur en retirant sa moyenne sur les trois extraits. Gardez un axe de taille un dans la référence.

In [16]:
exo_x = torch.tensor([[.1,1.5],[.2,4.5],[.15,2.5]])
exo_reference = ...  # Remplacer les points par votre réponse.
if exo_reference is Ellipsis or exo_reference is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_reference.shape) == (1,2)
    exo_centre=exo_x-exo_reference
    torch.testing.assert_close(exo_centre.mean(dim=0),torch.zeros(2),atol=1e-6,rtol=0)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Réduire l’axe 0 rassemble les extraits séparément pour chaque colonne. La référence se rediffuse sur les lignes.

</details>

### Exercice 2

Conservez les lignes dont la crête dépasse 2. Quelles cibles accompagnent les mesures dans le même ordre ?

In [17]:
exo_x = torch.tensor([[.1,1.5],[.2,4.5],[.15,2.5]])
exo_cibles = torch.tensor([0,1,0])
exo_masque = exo_x[:,1] > 2
exo_cibles_gardees = ...  # Remplacer les points par votre réponse.
if exo_cibles_gardees is Ellipsis or exo_cibles_gardees is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert torch.equal(exo_cibles_gardees,torch.tensor([1,0]))
    assert len(exo_cibles_gardees) == len(exo_x[exo_masque])
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Appliquez le même masque aux lignes et à leurs annotations.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_x = torch.tensor([[.1,1.5],[.2,4.5],[.15,2.5]])
exo_reference = exo_x.mean(dim=0,keepdim=True)
if exo_reference is Ellipsis or exo_reference is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_reference.shape) == (1,2)
    exo_centre=exo_x-exo_reference
    torch.testing.assert_close(exo_centre.mean(dim=0),torch.zeros(2),atol=1e-6,rtol=0)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_x = torch.tensor([[.1,1.5],[.2,4.5],[.15,2.5]])
exo_cibles = torch.tensor([0,1,0])
exo_masque = exo_x[:,1] > 2
exo_cibles_gardees = exo_cibles[exo_masque]
if exo_cibles_gardees is Ellipsis or exo_cibles_gardees is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert torch.equal(exo_cibles_gardees,torch.tensor([1,0]))
    assert len(exo_cibles_gardees) == len(exo_x[exo_masque])
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [20]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()